# Les agrégats respirent · *The aggregates breathe*

Notebook compagnon du chapitre **25. Masse monétaire M1, M2 : ce que ces agrégats mesurent vraiment** — [lire l'article](https://nmlab.io/ressources/masse-monetaire-m1-m2).
Companion notebook to chapter **25. Money Supply M1, M2: What These Aggregates Really Measure** — [read the article](https://nmlab.io/en/ressources/money-supply-m1-m2).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (les séries sont chargées dans build_figure)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    m2=load("M2SL","2014-06"); m2g=100*(m2/m2.shift(12)-1).dropna()
    m3=load("EA_M3","2014-06"); m3g=100*(m3/m3.shift(12)-1).dropna()
    fig=nm.figure(1010); ax=nm.axes(fig)
    ax.axhline(0,color=C["edge"],lw=1.5)
    ax.plot(m2g.index,m2g.values,color=C["blue"],lw=3,label=("M2 États-Unis" if lang=="fr" else "M2 United States"))
    ax.plot(m3g.index,m3g.values,color=C["amber"],lw=3,label=("M3 zone euro" if lang=="fr" else "M3 euro area"))
    d=dict(fr=("Les agrégats respirent — des deux côtés de l'Atlantique","Croissance sur un an de la masse monétaire large, en %.",
               "+26,8 % (fév. 2021)","Aucune cible ne les fixe plus ; ils restent un thermomètre du régime monétaire. Sources : FRED (M2), BCE (M3)."),
           en=("The aggregates breathe — on both sides of the Atlantic","Year-on-year growth of broad money, in %.",
               "+26.8% (Feb. 2021)","No target pins them down now; they remain a thermometer of the monetary regime. Sources: FRED (M2), ECB (M3)."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    xpk=m2g.idxmax()
    ax.annotate(t[2],xy=(xpk,m2g.max()),xytext=(pd.Timestamp("2016-02-01"),22),
                fontsize=17,color=C["blue"],fontweight="bold",va="center",
                arrowprops=dict(arrowstyle="-|>",color=C["blue"],lw=1.8))
    leg=ax.legend(loc="upper right",fontsize=18,frameon=True,facecolor=C["bg"],edgecolor=C["edge"])
    for txt in leg.get_texts(): txt.set_color(C["text"])
    ax.set_ylim(-8,30)
    nm.footer(fig,t[3]);
    return fig


build_figure(LANG)